# hub

> Publish an export to the Hugging Face Hub with a model card written from its `sysone.json` and training record, private by default; and move datasets through private Hub repos, the way data reaches a remote job.

In [ ]:
#| default_exp hub

In [ ]:
#| export
import json, re, tempfile
from pathlib import Path

from fastcore.basics import patch
from huggingface_hub import EvalResult, HfApi, ModelCard, ModelCardData, snapshot_download

from sysone.learner import Learner

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging, os, shutil
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.inference import Decider
from huggingface_hub import whoami

`Decider.load("owner/repo")` already reads a model from the Hub. This notebook covers the way there: `learn.push_to_hub(repo)` exports a learner and uploads the folder, with a `README.md` that the Hub shows as the model's page.

Three conventions keep a growing set of fine-tunes tidy:

- **Private by default.** A model goes public only when `private=False` is passed.
- **One repo per model, one commit per run.** Pushing a new run of the same model to the same repo keeps its history: the Hub's commit list becomes the list of runs, each with a one-line summary (encoder, regime, main metric, code commit). `revision="branch"` pushes to a branch instead, and `tag="v1"` marks a run worth keeping.
- **The card is written from the export.** Nothing is typed by hand but an optional description: the metadata block (library, base model, datasets, tags, and the metrics in the Hub's evaluation format) and the page itself come from `sysone.json` and its [training record](05a_record.ipynb), with any held-out evaluation recorded in it.

The fixtures: a tiny ModernBERT with LoRA, trained for one epoch and exported.

In [ ]:
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
learn = decision_learner(data, "fixtures/tiny-modernbert", train="lora", preset="cpu", logging_steps=1)
learn.fit_one_cycle(1, lr=1e-3)
out = learn.export(Path(tempfile.mkdtemp()) / "tiny-jev")

## The model card

`card_data` builds the metadata block. The base model is the encoder when it is a Hub id (a local folder is not), the datasets are the training data when it came from the Hub, and the tags say what the model is: `sysone`, `typed-decisions`, the modality, and how it was trained (`head-only`, `lora` or `full-finetune`). When the data has a Hub id, the validation metrics also go into the model index, which the Hub shows beside the card; there is no `pipeline_tag`, since the Hub's inference widgets cannot run a sysone model.

In [ ]:
#| export
def read_meta(path) -> dict:
    "An export's `sysone.json`"
    p = Path(path) / "sysone.json"
    if not p.exists(): raise FileNotFoundError(f"{path} is not a sysone export: it has no sysone.json")
    return json.loads(p.read_text())

def is_hub_id(s) -> bool:
    "Whether `s` names a Hub repo (`owner/name`) rather than a local folder"
    return isinstance(s, str) and re.fullmatch(r"[\w.-]+/[\w.-]+", s) is not None and not Path(s).exists()

SKIP_METRICS = {"epoch", "runtime", "samples_per_second", "steps_per_second", "n", "n_unanswerable", "seconds", "latency_p50_ms"}

def _metric_table(m:dict|None) -> dict:
    return {k: round(float(v), 4) for k, v in (m or {}).items()
            if k not in SKIP_METRICS and isinstance(v, (int, float)) and not isinstance(v, bool)}

def card_metrics(meta:dict) -> dict:
    "The validation metrics of an export, rounded, without timings and counts"
    return _metric_table(meta.get("metrics"))

def record_evaluation(path, metrics:dict, split:str="test", dataset:str|None=None, config:str|None=None, note:str|None=None) -> dict:
    "Add a held-out evaluation (what `sysone.evaluate.evaluate` returns) to an export's `sysone.json`, which its card reports first; one per dataset and split"
    path = Path(path); meta = read_meta(path)
    ev = {"split": split, "dataset": dataset, "config": config, "n": metrics.get("n"), "metrics": _metric_table(metrics), "note": note}
    meta["evaluations"] = [e for e in meta.get("evaluations") or [] if (e.get("dataset"), e.get("split")) != (dataset, split)] + [ev]
    (path / "sysone.json").write_text(json.dumps(meta, indent=2, default=str))
    return meta

def _training(meta:dict) -> dict: return meta.get("training") or {}

def hub_dataset(meta:dict) -> dict|None:
    "The Hub dataset an export trained on, from its training record, or None"
    src = (_training(meta).get("data") or {}).get("source") or {}
    return src if src.get("kind") == "hub" and src.get("id") else None

TASKS = {"text": "zero-shot-classification", "image": "zero-shot-image-classification", "multimodal": "zero-shot-image-classification"}
REGIME_TAGS = {"head": "head-only", "lora": "lora", "full": "full-finetune"}

def card_data(meta:dict, license:str|None=None, tags:list|None=None) -> ModelCardData:
    "The model card's metadata: library, base model, datasets, tags and, for Hub data, the validation metrics as model-index results"
    enc, ds = meta.get("encoder") or {}, hub_dataset(meta)
    regime = (meta.get("head") or {}).get("regime")
    tags = ["sysone", "typed-decisions", enc.get("modality") or "text", REGIME_TAGS.get(regime, regime)] + list(tags or [])
    results = [EvalResult(task_type=TASKS.get(enc.get("modality"), "zero-shot-classification"), dataset_type=ds["id"],
                          dataset_name=ds["id"].split("/")[-1], dataset_config=ds.get("config"),
                          dataset_split=ds.get("valid") or ds.get("split"), dataset_revision=ds.get("sha"),
                          metric_type=k, metric_value=v) for k, v in card_metrics(meta).items()] if ds else []
    results += [EvalResult(task_type=TASKS.get(enc.get("modality"), "zero-shot-classification"), dataset_type=e["dataset"],
                           dataset_name=e["dataset"].split("/")[-1], dataset_config=e.get("config"), dataset_split=e.get("split"),
                           metric_type=k, metric_value=v)
                for e in meta.get("evaluations") or [] if is_hub_id(e.get("dataset")) for k, v in (e.get("metrics") or {}).items()]
    return ModelCardData(library_name="sysone", base_model=enc.get("id") if is_hub_id(enc.get("id")) else None,
                         datasets=[ds["id"]] if ds else None, license=license, model_name=meta.get("name"),
                         tags=list(dict.fromkeys(t for t in tags if t)), eval_results=results or None)

`card_text` writes the page: how to use the model, its results, how it was trained (data, encoder, head, loss, every fit, hardware), and how to train it again (code commit, entry point, package versions). `record_evaluation` adds a held-out evaluation to the export, such as the test split scored by `sysone.evaluate.evaluate`, and the card reports it before the validation metrics. An export that holds an adapter alone says so, and which encoder `Decider.load` downloads for it; `description` adds a paragraph of the publisher's own. Its "Use it" section installs sysone (with the `multimodal` extra for an image model) and asks an example: a state and typed questions, passed to `dec.predict`. A text model gets a support ticket unless the publisher passes `example`, a dict with the `state`, the `questions` and, from a run of the model, its `answers`; or with `code` of its own, shown in their place.

In [ ]:
#| export
def _short(sha, n:int=8): return f"`{sha[:n]}`" if sha else "?"

def source_text(src:dict|None) -> str:
    "One line naming where the cases came from"
    if not src: return "not recorded"
    reg = f"the `{src['registry']}` dataset of sysone's registry, " if src.get("registry") else ""
    k = src.get("kind")
    if k == "hub":
        at = f" at {_short(src.get('sha'))}" if src.get("sha") else (f" at `{src['revision']}`" if src.get("revision") else "")
        return reg + f"[{src['id']}](https://huggingface.co/datasets/{src['id']})" + (f" (`{src['config']}`)" if src.get("config") else "") + at
    if k == "file": return reg + ", ".join(f"`{f['name']}` (SHA-256 {_short(f.get('sha256'), 12)})" for f in src.get("files", []))
    return reg + {"records": "records in memory", "dataframe": "a DataFrame"}.get(k, str(k))

def _num(x, fmt=".3g"): return format(x, fmt) if isinstance(x, (int, float)) else "?"

def _targets(t) -> str:
    "LoRA's target modules: a regular expression or a list of names"
    return f"`{t}`" if isinstance(t, str) else ", ".join(f"`{x}`" for x in t or [])

def _seconds(s) -> str:
    if not isinstance(s, (int, float)): return "?"
    return f"{s:.1f} s" if s < 120 else f"{s / 60:.1f} min" if s < 7200 else f"{s / 3600:.1f} h"

def fit_text(f:dict) -> str:
    "One line describing a fit from the training record"
    lr = f"lr {_num(f.get('lr_head'), '.2g')}" + (f" (encoder {_num(f.get('lr_encoder'), '.2g')})" if f.get("regime") != "head" else "")
    ep = f.get("epochs")
    return (f"{_num(ep, 'g')} epoch{'' if ep == 1 else 's'}, {f.get('schedule')}, {lr}, batch {f.get('batch_size')}×{f.get('grad_accum')}, "
            f"{f.get('precision')} on {f.get('device')}: {f.get('steps')} steps in {_seconds(f.get('seconds'))}, "
            f"peak memory {_num(f.get('peak_memory_gb'), 'g')} GB, final loss {_num(f.get('final_loss'), '.3f')}")

INSTALL = "git+https://github.com/sgaseretto/sysonelib"
CARD_EXAMPLE = {"state": {"from": "user@acme.com", "body": "I was charged twice for invoice 4411 and I am cancelling if this is not fixed today."},
                "questions": {"department": {"type": "choice", "instructions": "Which department should handle this?",
                                             "criteria": {"billing": "invoices, refunds", "technical": "bugs, outages"}},
                              "urgency": {"type": "score", "instructions": "How urgent is it?", "criteria": ["not urgent", "soon", "critical"]},
                              "churn_risk": {"type": "noul", "instructions": "Does the user threaten to leave?"}}}

def _one(x) -> str:
    if isinstance(x, dict): return "{" + ", ".join(f"{_one(k)}: {_one(v)}" for k, v in x.items()) + "}"
    if isinstance(x, (list, tuple)): return "[" + ", ".join(_one(v) for v in x) + "]"
    return json.dumps(x, ensure_ascii=False) if isinstance(x, str) else repr(x)

def _literal(x, indent:int=0, at:int=0, width:int=120) -> str:
    "`x` as Python source, the way black writes it: on one line when it fits from column `at`, else an item per line"
    one = _one(x)
    if at + len(one) < width or not isinstance(x, (dict, list, tuple)) or not x: return one
    pad = " " * (indent + 4)
    items = ([f"{pad}{_one(k)}: " + _literal(v, indent + 4, len(pad) + len(_one(k)) + 2, width) + "," for k, v in x.items()]
             if isinstance(x, dict) else [pad + _literal(v, indent + 4, len(pad), width) + "," for v in x])
    o, c = "{}" if isinstance(x, dict) else "[]"
    return o + "\n" + "\n".join(items) + "\n" + " " * indent + c

def _use_it(repo_id:str|None, enc:dict, example:dict|None) -> list:
    "The card's 'Use it': installing sysone, then a state and typed questions passed to `dec.predict`, and their answers when given"
    image = enc.get("modality") in ("image", "multimodal")
    pip = f'pip install "sysone[multimodal] @ {INSTALL}"' if image else f"pip install {INSTALL}"
    ex = example if example is not None else (None if image else CARD_EXAMPLE)
    lines = ["", "## Use it", "", "Install sysone from GitHub, in Python 3.11 or newer:", "", "```sh", pip, "```", ""]
    if ex is None:
        return lines + ["```python", "from sysone.inference import Decider", "", f'dec = Decider.load("{repo_id or "<this repo>"}")',
                        "dec.predict(state, questions)   # answers in the Jev schema", "```"]
    lines += ["A state is whatever the decision is about, as JSON-like data. Each question has a type: a `choice` among named options, "
              "a `score` on an ordered scale, or a `noul`, a statement that is true or false. The options are named when asking, so they "
              "can be new ones.", "", "```python", "from sysone.inference import Decider", "",
              f'dec = Decider.load("{repo_id or "<this repo>"}")', ""]
    lines += (ex["code"].strip().splitlines() if ex.get("code") else
              [f"state = {_literal(ex['state'], 0, 8)}", f"questions = {_literal(ex['questions'], 0, 12)}", "answers = dec.predict(state, questions)"])
    lines += ["```", ""]
    schema = ("`answers` holds an answer per question, in the Jev schema: a choice names the likeliest option and gives every option's "
              "probability, a score gives its expected level on the scale (0 for the first) and every level's probability, and a noul "
              "gives the probability that its statement is true; each says how confident it is.")
    if not ex.get("answers"): return lines + [schema]
    return lines + [schema + " This model's answers to the example:", "", "```python", _literal(ex["answers"]), "```"]

def _evaluation_text(e:dict) -> list:
    ds = e.get("dataset")
    where = (f"[{ds}](https://huggingface.co/datasets/{ds})'s" if is_hub_id(ds) else f"`{ds}`'s" if ds else "the") + f" `{e.get('split')}` split"
    return ["", f"On {where}" + (f" ({int(e['n']):,} decisions)" if e.get("n") else "") + (f", {e['note']}" if e.get("note") else "") + ":", "",
            "| metric | value |", "|---|---|"] + [f"| {k} | {v:.4g} |" for k, v in (e.get("metrics") or {}).items()]

def card_text(meta:dict, repo_id:str|None=None, adapter_only:bool=False, description:str|None=None, example:dict|None=None) -> str:
    "The model card's page, from an export's `sysone.json`; `adapter_only` when the export holds an adapter without the encoder's weights"
    tr, enc, head = _training(meta), meta.get("encoder") or {}, meta.get("head") or {}
    data, model, hw = tr.get("data") or {}, tr.get("model") or {}, tr.get("hardware") or {}
    regime = head.get("regime")
    lines = [f"# {meta.get('name') or 'sysone model'}", "",
             f"A typed-decision model made with [sysone](https://github.com/sgaseretto/sysonelib): the encoder `{enc.get('id')}` "
             f"({REGIME_TAGS.get(regime, regime)}) under a {head.get('layers')}-layer `{head.get('scorer')}` head, trained on "
             f"{source_text(data.get('source'))}."]
    if description: lines += ["", description]
    if adapter_only:
        lines += ["", f"The repo holds the head and a LoRA adapter, not the encoder's weights: `Decider.load` downloads `{enc.get('id')}`"
                      + (f" at commit {_short(enc.get('revision'), 10)}" if enc.get("revision") else "") + " from the Hub and puts the adapter on it."]
    lines += _use_it(repo_id, enc, example)
    if enc.get("weights") == "laya": lines += ["", "Laya's runtime loads it as well: the folder holds `rl_agent_config.json` and a single `model.safetensors`."]
    m, evs = card_metrics(meta), meta.get("evaluations") or []
    if m or evs: lines += ["", "## Results"]
    for e in evs: lines += _evaluation_text(e)
    if m:
        n = (meta.get("metrics") or {}).get("n")
        lines += ["", f"On the validation split{f' ({int(n):,} decisions)' if n else ''}{', during training' if evs else ''}:", "", "| metric | value |", "|---|---|"]
        lines += [f"| {k} | {v:.4g} |" for k, v in m.items()]
    if meta.get("temperature"): lines += ["", "Calibration temperatures: " + ", ".join(f"{k} {v:.3g}" for k, v in meta["temperature"].items()) + "."]
    if tr:
        t, lora = tr.get("training") or {}, model.get("lora")
        splits = ", ".join(f"{v['cases']} {k}" for k, v in (data.get("splits") or {}).items())
        acc, host, run = hw.get("accelerator") or {}, hw.get("host") or {}, hw.get("run") or {}
        rows = [("Data", f"{source_text(data.get('source'))}; {splits} cases"),
                ("Encoder", f"`{enc.get('id')}`, {REGIME_TAGS.get(regime, regime)}" + (f" (LoRA r={lora.get('r')}, α={lora.get('lora_alpha')}, on {_targets(lora.get('target_modules'))})" if lora else "")),
                ("Head", f"{head.get('layers')} layers, `{head.get('scorer')}` scorer, readout `{head.get('readout')}`"),
                ("Loss", f"`{t.get('loss')}`, preset `{t.get('preset')}`, seed {t.get('seed')}")]
        prev = tr.get("resumed_from")
        if prev:                    # trained further from an export (load_learner)
            pt = prev.get("training") or {}
            pf, prun = len((pt.get("training") or {}).get("fits") or []), (pt.get("hardware") or {}).get("run") or {}
            rows.append(("Continued from", f"`{prev.get('source') or prev.get('name')}`, after {pf} fit{'s' if pf != 1 else ''} on a "
                                           f"{prun.get('backend') or 'local'} run" + (f" (job `{prun['job']}`)" if prun.get("job") else "")))
        rows += [(f"Fit {i + 1}", fit_text(f)) for i, f in enumerate(t.get("fits") or [])]
        rows += [("Machine", f"{host.get('cpu')}, {host.get('cores')} cores, {host.get('memory_gb')} GB; accelerator {acc.get('kind')}"
                  + (f" ({', '.join(str(d) for d in acc.get('devices') or [])})" if acc.get("devices") and acc.get("kind") != "mps" else "")
                  + f"; a {run.get('backend')} run" + (f" (job `{run.get('job')}`)" if run.get("job") else ""))]
        lines += ["", "## Training", "", "| | |", "|---|---|"] + [f"| {k} | {v} |" for k, v in rows]
        code, env = tr.get("code") or {}, tr.get("environment") or {}
        entry = code.get("entry") or {}
        lines += ["", "## Reproduce", ""]
        for label, g in (("Code", code.get("git") or {}), ("sysone", code.get("sysone_git") or {})):     # sysone's checkout, when it is another
            if g.get("commit"):
                lines.append(f"- {label}: `{g.get('repo')}` at commit {_short(g.get('commit'), 10)} ({g.get('branch')})"
                             + (f", with uncommitted changes in {g.get('n_changed')} files (diff SHA-256 {_short(g.get('diff_sha256'), 12)})" if g.get("dirty") else ""))
        if entry: lines.append(f"- Ran: `{entry.get('name')}`")
        pk = env.get("packages") or {}
        lines.append(f"- Python {env.get('python')}, " + ", ".join(f"{k} {v}" for k, v in pk.items()) + "; `environment.txt` lists every package")
        trk = (tr.get("tracking") or {}).get("mlflow")
        if trk: lines.append(f"- MLflow run `{trk.get('run_id')}` ({trk.get('tracking_uri')})")
    return "\n".join(lines) + "\n"

def model_card(path, repo_id:str|None=None, license:str|None=None, tags:list|None=None, description:str|None=None,
               example:dict|None=None) -> ModelCard:
    "The Hub model card of an export: metadata from `card_data`, page from `card_text`"
    meta = read_meta(path)
    alone = (meta.get("encoder") or {}).get("weights") == "adapter" and not (Path(path) / "encoder").exists()
    return ModelCard(f"---\n{card_data(meta, license, tags).to_yaml()}\n---\n\n{card_text(meta, repo_id, alone, description, example)}")

In [ ]:
card = model_card(out, "me/tiny-jev")
test_eq((card.data.library_name, card.data.base_model, card.data.datasets), ("sysone", None, None))   # a local encoder, local data
assert {"sysone", "typed-decisions", "text", "lora"} <= set(card.data.tags)
assert 'Decider.load("me/tiny-jev")' in card.text and "## Training" in card.text and "Fit 1" in card.text
assert f"on `{learn.model.encoder.peft_config['default'].target_modules}`" in card.text   # the regular expression, whole
print(card.text)

A model trained on a Hub dataset over a Hub encoder gets both into its metadata, and its metrics into the model index:

In [ ]:
hubbed = Path(tempfile.mkdtemp()) / "hub-jev"; hubbed.mkdir()
meta = read_meta(out)
meta["encoder"]["id"] = "answerdotai/ModernBERT-large"
meta["training"]["data"]["source"] = {"kind": "hub", "id": "LocalLLaMA/typed-decisions", "config": "all", "split": "train",
                                      "valid": "test", "revision": None, "sha": "0123456789abcdef0123"}
(hubbed / "sysone.json").write_text(json.dumps(meta))
hcard = model_card(hubbed, "me/hub-jev", license="apache-2.0")
test_eq((hcard.data.base_model, hcard.data.datasets, hcard.data.license), ("answerdotai/ModernBERT-large", ["LocalLLaMA/typed-decisions"], "apache-2.0"))
res = hcard.data.to_dict()["model-index"][0]["results"]
test_eq((len(res), len(res[0]["metrics"])), (1, len(card_metrics(meta))))   # one task and dataset, a metric each
test_eq((res[0]["dataset"]["type"], res[0]["dataset"]["split"], res[0]["dataset"]["revision"]), ("LocalLLaMA/typed-decisions", "test", "0123456789abcdef0123"))
test_eq(ModelCard(str(hcard)).data.base_model, "answerdotai/ModernBERT-large")   # the text parses back
print(str(hcard).split("---")[1])

A held-out evaluation recorded in the export comes first, under its dataset and split, and goes into the model index too; recording the same split again replaces it. An adapter exported alone is named as such:

In [ ]:
ev = Path(tempfile.mkdtemp()) / "evaluated"; shutil.copytree(hubbed, ev)
record_evaluation(ev, {"accuracy": 0.5, "ece": 0.1, "n": 40, "latency_p50_ms": 3.0}, dataset="LocalLLaMA/typed-decisions", note="calibrated")
ecard = model_card(ev, "me/evaluated", description="A tiny model.")
assert "On [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)'s `test` split (40 decisions), calibrated:" in ecard.text
assert "A tiny model." in ecard.text and "latency" not in ecard.text and "on the validation split" not in ecard.text.lower().split("calibrated:")[0]
test_eq(len(ecard.data.to_dict()["model-index"][0]["results"]), 2)       # the validation metrics, and the test split's
record_evaluation(ev, {"accuracy": 0.6, "n": 40}, dataset="LocalLLaMA/typed-decisions")
test_eq([e["metrics"] for e in read_meta(ev)["evaluations"]], [{"accuracy": 0.6}])
alone = learn.export(Path(tempfile.mkdtemp()) / "alone", merge=False, bundle=False)
assert "not the encoder's weights" in model_card(alone).text and "not the encoder's weights" not in card.text
test_eq((alone / "adapter" / "README.md").exists(), False)     # peft's placeholder card stays out of the export

"Use it" installs sysone and asks an example. A text model gets the support ticket unless the publisher passes one, with the answers of a run of the model when there are some; an image model gets the `multimodal` extra and no example, since a state of its own holds an image:

In [ ]:
assert "pip install git+https://github.com/sgaseretto/sysonelib" in card.text and "answers = dec.predict(state, questions)" in card.text
assert '"churn_risk": {"type": "noul", "instructions": "Does the user threaten to leave?"}' in card.text
ex = {"state": {"ticket": "My card was declined."}, "questions": {"declined": {"type": "noul", "instructions": "Was a card declined?"}},
      "answers": {"declined": {"type": "noul", "noul": 0.9, "confidence": 0.9}}}
etext = card_text(read_meta(out), "me/tiny-jev", example=ex)
assert 'state = {"ticket": "My card was declined."}' in etext and '{"declined": {"type": "noul", "noul": 0.9, "confidence": 0.9}}' in etext
assert "churn_risk" not in etext
ctext = card_text(read_meta(out), "me/tiny-jev", example={"code": "answers = dec.predict(page, steps)\nprint(answers)"})
assert "answers = dec.predict(page, steps)\nprint(answers)\n```" in ctext and "state = " not in ctext
img = read_meta(out); img["encoder"]["modality"] = "image"
assert 'pip install "sysone[multimodal] @ git+https://github.com/sgaseretto/sysonelib"' in card_text(img) and "state = " not in card_text(img)
test_eq(_literal({"a": [1, "b"]}), '{"a": [1, "b"]}')
test_eq(_literal({"a": ["x" * 20, "y" * 20]}, width=30), '{\n    "a": [\n        "' + "x" * 20 + '",\n        "' + "y" * 20 + '",\n    ],\n}')
print(card_text(read_meta(out), "me/tiny-jev").split("## Results")[0])

## Pushing

`push_to_hub` writes the card into the folder as `README.md`, makes the repo if it does not exist (private unless told otherwise), uploads the folder as one commit, and tags it when asked. `dry_run=True` returns what it would do, the card included, and touches nothing. `learn.push_to_hub(repo)` exports first, to a temporary folder named after the repo.

In [ ]:
#| export
def commit_summary(meta:dict) -> str:
    "A one-line commit message: the model, its encoder and regime, its main metric and the code commit"
    enc, regime = (meta.get("encoder") or {}).get("id") or "encoder", (meta.get("head") or {}).get("regime")
    evs = meta.get("evaluations") or []
    m = (evs[0].get("metrics") or {}) if evs else card_metrics(meta)     # a held-out evaluation before the validation metrics
    main = next((f"{k} {m[k]:.3f}" + (f" ({evs[0].get('split')})" if evs else "") for k in ("macro_f1", "accuracy") if k in m), None)
    c = _training(meta).get("code") or {}
    g = c.get("git") or c.get("sysone_git") or {}       # a job's entry may sit outside any repository
    code = f"code {g['commit'][:7]}{'+' if g.get('dirty') else ''}" if g.get("commit") else None
    return " · ".join(x for x in (f"{meta.get('name')}: {str(enc).split('/')[-1]} {REGIME_TAGS.get(regime, regime)}", main, code) if x)

def push_to_hub(path, repo_id:str, private:bool=True, card:bool=True, license:str|None=None, tags:list|None=None,
                commit_message:str|None=None, revision:str|None=None, tag:str|None=None, token=None, dry_run:bool=False,
                description:str|None=None, example:dict|None=None) -> dict:
    "Upload an export folder to a Hub model repo, private by default, with a model card written from its sysone.json"
    path = Path(path); meta = read_meta(path)
    text = str(model_card(path, repo_id, license, tags, description, example)) if card else None
    files = sorted(str(p.relative_to(path)) for p in path.rglob("*") if p.is_file())
    plan = {"repo": repo_id, "private": private, "revision": revision or "main", "tag": tag,
            "message": commit_message or commit_summary(meta), "files": sorted(set(files) | ({"README.md"} if card else set()))}
    if dry_run: return plan | {"card": text}
    if card: (path / "README.md").write_text(text)
    api = HfApi(token=token)
    api.create_repo(repo_id, private=private, exist_ok=True)
    if revision not in (None, "main"): api.create_branch(repo_id, branch=revision, exist_ok=True)
    info = api.upload_folder(repo_id=repo_id, folder_path=path, commit_message=plan["message"], revision=revision)
    if tag: api.create_tag(repo_id, tag=tag, revision=info.oid)
    return plan | {"commit": info.oid, "url": info.commit_url}

@patch
def push_to_hub(self:Learner, repo_id:str, path=None, **kw) -> dict:
    "Export the learner (to a temporary folder named after the repo, unless `path` is given) and push the export to the Hub"
    path = self.export(Path(path) if path else Path(tempfile.mkdtemp()) / repo_id.split("/")[-1])
    return push_to_hub(path, repo_id, **kw)

In [ ]:
plan = push_to_hub(out, "me/tiny-jev", tag="v1", dry_run=True)
test_eq((plan["private"], plan["revision"], plan["tag"]), (True, "main", "v1"))
assert {"README.md", "sysone.json", "environment.txt", "head.safetensors"} <= set(plan["files"])
assert not (out / "README.md").exists()          # a dry run writes nothing
assert plan["message"].startswith("tiny-jev: tiny-modernbert lora · accuracy")
m2 = read_meta(out); m2["training"]["code"] = {"git": None, "sysone_git": {"repo": "sysonelib", "commit": "abcdef123456", "branch": "main", "dirty": True,
                                                                          "n_changed": 2, "diff_sha256": "0f" * 32}}
m3 = read_meta(out); m3["training"]["resumed_from"] = {"name": "first", "source": "first",
                                                      "training": {"hardware": {"run": {"backend": "colab", "job": "colab-1"}}, "training": {"fits": [{}, {}]}}}
assert "| Continued from | `first`, after 2 fits on a colab run (job `colab-1`) |" in card_text(m3)
assert commit_summary(m2).endswith("code abcdef1+") and "- sysone: `sysonelib` at commit `abcdef1234` (main), with uncommitted changes in 2 files" in card_text(m2)
plan["message"]

In [ ]:
test_eq(learn.push_to_hub("me/other-name", dry_run=True)["files"], plan["files"])
test_eq(commit_summary(read_meta(ev)).split(" · ")[1], "accuracy 0.600 (test)")      # a held-out evaluation names the run
test_fail(lambda: push_to_hub(Path(tempfile.mkdtemp()), "me/x", dry_run=True), contains="no sysone.json")

## Data through the Hub

A remote job reads its data from where it can reach it. For Colab and Hugging Face's own machines, that is the Hub: `push_data` uploads a file or a folder to a private dataset repo, and the job reads it back with `pull_data` (or `TypedDecisions.from_hub` when the data is in the typed-decisions format). Kaggle and SageMaker have their own stores (Kaggle datasets and S3), which [cloud](40_cloud.ipynb) uses for them.

In [ ]:
#| export
def push_data(path, repo_id:str, private:bool=True, commit_message:str|None=None, token=None, dry_run:bool=False) -> dict:
    "Upload a file or a folder to a Hub dataset repo, private by default"
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    files = sorted(str(p.relative_to(path)) for p in path.rglob("*") if p.is_file()) if path.is_dir() else [path.name]
    plan = {"repo": repo_id, "private": private, "files": files, "message": commit_message or f"sysone data: {path.name}"}
    if dry_run: return plan
    api = HfApi(token=token)
    api.create_repo(repo_id, repo_type="dataset", private=private, exist_ok=True)
    if path.is_dir(): info = api.upload_folder(repo_id=repo_id, repo_type="dataset", folder_path=path, commit_message=plan["message"])
    else: info = api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name, repo_id=repo_id, repo_type="dataset",
                                 commit_message=plan["message"])
    return plan | {"commit": info.oid, "url": info.commit_url}

def pull_data(repo_id:str, revision:str|None=None, dest=None, token=None) -> Path:
    "Download a Hub dataset repo (at `revision`) and return its folder"
    return Path(snapshot_download(repo_id, repo_type="dataset", revision=revision, local_dir=dest, token=token))

In [ ]:
test_eq(push_data("fixtures/typed_decisions_tiny.jsonl", "me/tiny-data", dry_run=True)["files"], ["typed_decisions_tiny.jsonl"])
test_eq(push_data("fixtures/tiny-modernbert", "me/tiny-encoder", dry_run=True)["private"], True)

## On the real Hub

These cells need a Hugging Face login with write access (`hf auth login`, or `sysone.auth.login("hf")` in a notebook). They are flagged `hub`, so `nbdev-test` skips them unless run with `--flags hub`. They push the tiny export to a private repo of the logged-in account, read it back with `Decider.load`, and check that it answers as the local folder does.

In [ ]:
#| hub
repo = f"{whoami()['name']}/sysone-hub-test"
pushed = push_to_hub(out, repo)
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(Decider.load(repo, device="cpu").predict(rec["state"], rec["questions"]), Decider.load(out, device="cpu").predict(rec["state"], rec["questions"]))
assert ModelCard.load(repo).data.library_name == "sysone"
pushed["url"]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()